# 7.3 — The Convolution Operation

A convolution turns one small grid of weights into a reusable pattern detector: place the kernel on a local image window, multiply matching entries, sum them, then slide to the next location. In this lesson, we build that operation from scratch with NumPy so stride, padding, weight sharing, channels, and receptive field growth are visible as ordinary array arithmetic rather than hidden library calls.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build convolution one idea at a time. Run each cell in order and read the printed intermediate values — every output is one small local dot product, and every visualization is there to make the sliding-window geometry inspectable. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, dot products, padding, and numerical checks.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for any synthetic images.

### 1. Images are grids; kernels are small stencils

A grayscale image is just a matrix of intensities. A kernel is a much smaller matrix whose entries say what local pattern to reward or penalize. The important design choice is locality: instead of assigning one weight to every pixel in the whole image, the kernel only looks at a tiny neighborhood at a time.

In [ ]:
image_w = np.array([[0., 0., 0., 0., 0.],
                    [0., 1., 1., 1., 0.],
                    [0., 1., 3., 1., 0.],
                    [0., 1., 1., 1., 0.],
                    [0., 0., 0., 0., 0.]])  # a bright blob in the center.
kernel_w = np.array([[0., 1., 0.],
                     [1., -4., 1.],
                     [0., 1., 0.]])  # center-surround stencil.
print("image shape:", image_w.shape, "kernel shape:", kernel_w.shape)
print("kernel sum:", kernel_w.sum())
assert image_w.shape == (5, 5) and kernel_w.shape == (3, 3)

▶ What you'll see: a 5×5 image and a 3×3 kernel; the kernel has one negative center and four positive neighbors.

In [ ]:
fig_w, ax_w = plt.subplots(1, 2, figsize=(6, 2.8))
ax_w[0].imshow(image_w, cmap="gray", vmin=0, vmax=3); ax_w[0].set_title("image grid")
ax_w[1].imshow(kernel_w, cmap="coolwarm", vmin=-4, vmax=4); ax_w[1].set_title("kernel weights")
for a_w in ax_w:
    a_w.set_xticks(range(a_w.images[0].get_array().shape[1])); a_w.set_yticks(range(a_w.images[0].get_array().shape[0]))
plt.suptitle("1: a big grid and a small stencil"); plt.show()

▶ What you'll see: the image is spatial data; the kernel is the same small stencil that will be reused everywhere.

*Why it's done this way: locality encodes the image assumption that nearby pixels matter together. A 3×3 stencil can test a tiny pattern without spending a separate parameter for every absolute pixel position.*

### 2. One output number is one dot product

At one location, convolution is no more mysterious than a dot product. We cut out a kernel-sized patch, multiply it elementwise by the kernel, and sum the products. Deep-learning libraries usually call this operation convolution even though the kernel is not flipped; mathematically, it is cross-correlation.

In [ ]:
patch_w = image_w[1:4, 1:4]  # the 3×3 window under the kernel at the center.
products_w = patch_w * kernel_w  # coordinatewise multiplication.
response_w = float(products_w.sum())  # one scalar feature response.
print("patch:\n", patch_w)
print("elementwise products:\n", products_w)
print("response:", response_w)
assert response_w == -8.0

▶ What you'll see: the bright center is multiplied by −4, so this center-surround kernel responds strongly negative.

In [ ]:
plt.figure(figsize=(3.6, 3))
plt.imshow(products_w, cmap="coolwarm", vmin=-12, vmax=3)
plt.colorbar(label="patch × kernel")
plt.title("2: terms that sum to one response")
plt.show()

▶ What you'll see: the largest negative term is the center product, which dominates the final sum.

*Why it's done this way: the dot product is a similarity score between the local patch and the kernel. Positive matching entries raise the response; negative weights subtract evidence, so the kernel can encode contrasts such as edge or center-surround patterns.*

### 3. Sliding the same dot product builds a feature map

A feature map is the collection of responses from every valid landing position. We do not learn a new kernel for each location; we slide the same weights and repeat the same dot product. That repeated reuse is the operation's defining geometry.

In [ ]:
def conv2d_valid_w(x_w, k_w):
    h_w, w_w = x_w.shape
    kh_w, kw_w = k_w.shape
    out_w = np.zeros((h_w - kh_w + 1, w_w - kw_w + 1))
    for i_w in range(out_w.shape[0]):
        for j_w in range(out_w.shape[1]):
            out_w[i_w, j_w] = np.sum(x_w[i_w:i_w + kh_w, j_w:j_w + kw_w] * k_w)
    return out_w

feature_w = conv2d_valid_w(image_w, kernel_w)
print("feature map shape:", feature_w.shape)
print(feature_w)
assert feature_w.shape == (3, 3)
assert feature_w[1, 1] == -8.0

▶ What you'll see: the 5×5 image with a 3×3 kernel produces a 3×3 valid feature map.

In [ ]:
plt.figure(figsize=(3.8, 3))
plt.imshow(feature_w, cmap="coolwarm")
plt.colorbar(label="kernel response")
plt.title("3: sliding responses")
plt.show()

▶ What you'll see: the center location has the strongest negative response because the bright center sits under the negative kernel weight.

*Why it's done this way: a feature map preserves location. If a pattern is found at row 2, column 3, the response appears at the corresponding output location instead of being collapsed into one global number.*

### 4. Weight sharing gives translation equivariance

Because every location uses the same kernel, shifting the input shifts the feature map. This property is translation equivariance: the response moves with the pattern. It is not magic; it is a direct consequence of reusing the same stencil at every landing position.

In [ ]:
image_shift_w = np.zeros_like(image_w)
image_shift_w[1:4, 2:5] = image_w[1:4, 1:4]  # shift the bright 3×3 block one column right.
feature_shift_w = conv2d_valid_w(image_shift_w, kernel_w)
print("original feature:\n", feature_w)
print("shifted feature:\n", feature_shift_w)
assert feature_shift_w[1, 2] == feature_w[1, 1]

▶ What you'll see: the distinctive response that was centered moves one output column to the right.

In [ ]:
fig_w, ax_w = plt.subplots(1, 2, figsize=(6, 2.8))
ax_w[0].imshow(feature_w, cmap="coolwarm"); ax_w[0].set_title("original response")
ax_w[1].imshow(feature_shift_w, cmap="coolwarm"); ax_w[1].set_title("shifted response")
plt.suptitle("4: the feature moves with the input"); plt.show()

▶ What you'll see: the response pattern is not relearned; it is relocated.

*Why it's done this way: images contain the same local evidence in many places. Weight sharing deliberately forces one detector to mean the same thing everywhere, which is why the model can recognize a feature in the top-left or bottom-right with the same parameters.*

### 5. Padding and stride decide which windows exist

Valid convolution shrinks maps because the kernel cannot center on the border without falling off the image. Padding adds a border so edge pixels can participate; stride skips landing positions to reduce resolution. The shape formula is

$$H_{out}=\left\lfloor\frac{H+2p-k}{s}\right\rfloor+1,$$

and the same calculation applies to width.

In [ ]:
def conv2d_w(x_w, k_w, stride_w=1, pad_w=0):
    xp_w = np.pad(x_w, ((pad_w, pad_w), (pad_w, pad_w)), mode="constant")
    kh_w, kw_w = k_w.shape
    oh_w = (xp_w.shape[0] - kh_w) // stride_w + 1
    ow_w = (xp_w.shape[1] - kw_w) // stride_w + 1
    out_w = np.zeros((oh_w, ow_w))
    for i_w in range(oh_w):
        for j_w in range(ow_w):
            r_w, c_w = i_w * stride_w, j_w * stride_w
            out_w[i_w, j_w] = np.sum(xp_w[r_w:r_w + kh_w, c_w:c_w + kw_w] * k_w)
    return out_w

shape_valid_w = conv2d_w(image_w, kernel_w, stride_w=1, pad_w=0).shape
shape_pad_w = conv2d_w(image_w, kernel_w, stride_w=1, pad_w=1).shape
shape_stride_w = conv2d_w(image_w, kernel_w, stride_w=2, pad_w=0).shape
print("valid:", shape_valid_w, "padded:", shape_pad_w, "stride-2:", shape_stride_w)
assert shape_valid_w == (3, 3) and shape_pad_w == (5, 5) and shape_stride_w == (2, 2)

▶ What you'll see: padding preserves the 5×5 size here, while stride 2 shrinks the map to 2×2.

In [ ]:
sizes_w = [shape_valid_w[0], shape_pad_w[0], shape_stride_w[0]]
plt.figure(figsize=(4.4, 3))
plt.bar(["valid", "pad=1", "stride=2"], sizes_w, color=["gray", "seagreen", "orange"])
plt.ylabel("output height")
plt.title("5: shape choices are explicit")
plt.show()

▶ What you'll see: the output size changes exactly as the formula predicts.

*Why it's done this way: padding is a boundary decision and stride is a sampling decision. Both leave the local dot-product definition unchanged, but they change the grid of locations where that dot product is evaluated.*

### 6. Parameter sharing makes convolution cheap

A dense layer from a 5×5 image to a 5×5 output would need a different weight for every input-output pair. A 3×3 convolution uses only 9 weights per output channel, reused at every spatial position. That enormous reduction is the practical reason CNNs scale to images.

In [ ]:
h_w, w_w = 5, 5
k_w = 3
conv_params_w = k_w * k_w
dense_params_w = (h_w * w_w) * (h_w * w_w)
print("conv weights:", conv_params_w)
print("dense weights for 25→25:", dense_params_w)
print("compression factor:", round(dense_params_w / conv_params_w, 2))
assert conv_params_w == 9 and dense_params_w == 625

▶ What you'll see: 9 shared convolution weights replace 625 dense weights in this tiny example.

In [ ]:
plt.figure(figsize=(4.2, 3))
plt.bar(["3×3 conv", "25→25 dense"], [conv_params_w, dense_params_w], color=["teal", "crimson"])
plt.yscale("log")
plt.ylabel("parameters, log scale")
plt.title("6: sharing collapses parameter count")
plt.show()

▶ What you'll see: even on a log scale, the dense layer spends far more parameters.

*Why it's done this way: sharing is a modeling assumption as well as an efficiency trick. We assume the same local pattern detector is useful everywhere, so the model should spend data learning one detector well instead of relearning it separately at every pixel.*

### 7. Channels and multiple kernels make feature volumes

A color image has several channels, and a CNN layer usually learns many kernels. One output channel is formed by summing dot products across all input channels; multiple kernels create multiple output feature maps stacked along a new channel axis.

In [ ]:
rgb_w = np.zeros((4, 4, 3))
rgb_w[:, :, 0] = np.eye(4)  # red diagonal.
rgb_w[:, :, 1] = np.fliplr(np.eye(4))  # green anti-diagonal.
rgb_w[:, :, 2] = 0.5  # blue background.
filter_w = np.zeros((2, 2, 3))
filter_w[:, :, 0] = np.array([[1., 0.], [0., 1.]])
filter_w[:, :, 1] = -np.array([[0., 1.], [1., 0.]])
filter_w[:, :, 2] = 0.25
print("input shape:", rgb_w.shape, "filter shape:", filter_w.shape)
assert rgb_w.shape == (4, 4, 3) and filter_w.shape == (2, 2, 3)

▶ What you'll see: the filter has a 2×2 spatial footprint and one 2×2 slice per input channel.

In [ ]:
def conv3d_one_filter_w(x_w, f_w):
    oh_w = x_w.shape[0] - f_w.shape[0] + 1
    ow_w = x_w.shape[1] - f_w.shape[1] + 1
    out_w = np.zeros((oh_w, ow_w))
    for i_w in range(oh_w):
        for j_w in range(ow_w):
            out_w[i_w, j_w] = np.sum(x_w[i_w:i_w + f_w.shape[0], j_w:j_w + f_w.shape[1], :] * f_w)
    return out_w

channel_feature_w = conv3d_one_filter_w(rgb_w, filter_w)
print(np.round(channel_feature_w, 2))
assert channel_feature_w.shape == (3, 3)

▶ What you'll see: each output value combines evidence from red, green, and blue at the same spatial window.

In [ ]:
filter2_w = -filter_w
volume_w = np.stack([conv3d_one_filter_w(rgb_w, filter_w), conv3d_one_filter_w(rgb_w, filter2_w)], axis=-1)
print("feature volume shape:", volume_w.shape)
assert volume_w.shape == (3, 3, 2)
plt.figure(figsize=(4, 3))
plt.imshow(volume_w[:, :, 0], cmap="coolwarm")
plt.colorbar(label="filter 0 response")
plt.title("7: one output channel")
plt.show()

▶ What you'll see: one kernel gives one map; stacking two kernels gives a 3×3×2 feature volume.

*Why it's done this way: channels let the detector combine different measured quantities at the same location, and multiple kernels let the layer ask many local questions in parallel.*

### 8. Stacking layers grows the receptive field

One 3×3 convolution is deliberately near-sighted: one output depends on only a 3×3 input patch. Stack layers, and the final unit depends on a larger region because its inputs already summarize neighborhoods. With stride 1 and 3×3 kernels, the receptive field grows as $1+2L$ after $L$ layers.

In [ ]:
layers_w = np.arange(1, 6)
rf_w = 1 + 2 * layers_w
print("layers:", layers_w)
print("receptive-field widths:", rf_w)
assert rf_w[0] == 3 and rf_w[1] == 5 and rf_w[-1] == 11

▶ What you'll see: two 3×3 layers see a 5×5 region, and five layers see an 11×11 region.

In [ ]:
plt.figure(figsize=(4.2, 3))
plt.plot(layers_w, rf_w, marker="o", color="purple")
plt.xlabel("number of 3×3 layers")
plt.ylabel("receptive-field width")
plt.title("8: depth expands context")
plt.show()

▶ What you'll see: receptive field grows linearly with depth for ordinary stride-1 3×3 layers.

*Why it's done this way: small kernels keep each layer cheap and local, while composition lets deeper networks integrate larger context without paying for one huge kernel at the first layer.*

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each convolution mechanic by hand.** Separate from the walkthrough
> above, here is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each
> toy prints intermediates with real `# ->` values, draws one picture, and checks the result.

### ✍️ Toy 1 · One convolution response is a local dot product

A kernel-sized patch is multiplied entrywise by a kernel, and the products sum to one scalar response.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)                 # seeded generator for this toy
t1_patch = np.array([[1., 2., 0.],
                     [0., 3., 1.],
                     [2., 1., 0.]])               # one 3x3 image patch
print("patch:", t1_patch.tolist())               # -> [[1.0, 2.0, 0.0], [0.0, 3.0, 1.0], [2.0, 1.0, 0.0]]
t1_kernel = np.array([[0., 1., 0.],
                      [1., -4., 1.],
                      [0., 1., 0.]])              # center-surround stencil
print("kernel:", t1_kernel.tolist())             # -> [[0.0, 1.0, 0.0], [1.0, -4.0, 1.0], [0.0, 1.0, 0.0]]
t1_products = t1_patch * t1_kernel               # -> [[0,2,0],[0,-12,1],[0,1,0]]
print("products:", t1_products.tolist())         # -> [[0.0, 2.0, 0.0], [0.0, -12.0, 1.0], [0.0, 1.0, 0.0]]
t1_response = float(t1_products.sum())           # -> -8.0
print("response:", t1_response)                  # -> -8.0
assert t1_response == -8.0

plt.figure(figsize=(3.6, 2.8))
plt.imshow(t1_products, cmap="coolwarm", vmin=-12, vmax=2)
plt.colorbar(label="patch × kernel")
plt.title("Toy 1 · products sum to -8")
plt.show()

▶ What you'll see: the negative center product dominates the sum, giving response `-8`.

### ✍️ Toy 2 · Sliding the same dot product builds a feature map

Valid convolution repeats the same patch score at every location where the kernel fully fits.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)                 # seeded generator for this toy
t2_img = np.array([[1., 2., 0., 1.],
                   [0., 3., 1., 2.],
                   [2., 1., 0., 0.],
                   [1., 0., 2., 3.]])             # -> 4x4 image
print("image:", t2_img.tolist())                 # -> [[1.0, 2.0, 0.0, 1.0], [0.0, 3.0, 1.0, 2.0], [2.0, 1.0, 0.0, 0.0], [1.0, 0.0, 2.0, 3.0]]
t2_kernel = np.array([[1., 0.], [0., -1.]])      # -> top-left minus bottom-right
print("kernel:", t2_kernel.tolist())             # -> [[1.0, 0.0], [0.0, -1.0]]
t2_feature = np.zeros((3, 3))                    # -> valid output shape
for t2_i in range(3):
    for t2_j in range(3):
        t2_feature[t2_i, t2_j] = np.sum(t2_img[t2_i:t2_i+2, t2_j:t2_j+2] * t2_kernel)
print("feature map:", t2_feature.tolist())       # -> [[-2.0, 1.0, -2.0], [-1.0, 3.0, 1.0], [2.0, -1.0, -3.0]]
assert t2_feature.shape == (3, 3) and t2_feature[1, 1] == 3.0

fig, t2_ax = plt.subplots(1, 2, figsize=(5.6, 2.5))
t2_ax[0].imshow(t2_img, cmap="viridis")
t2_ax[0].set_title("input")
t2_ax[1].imshow(t2_feature, cmap="coolwarm")
t2_ax[1].set_title("valid feature map")
for t2_a in t2_ax:
    t2_a.axis("off")
plt.suptitle("Toy 2 · slide one kernel")
plt.show()

▶ What you'll see: a 4×4 image with a 2×2 kernel produces a 3×3 response map.

### ✍️ Toy 3 · im2col turns sliding windows into matrix multiplication

The same sliding computation can be rearranged into one window matrix times one flattened kernel.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)                 # seeded generator for this toy
t3_img = np.array([[1., 2., 0., 1.],
                   [0., 3., 1., 2.],
                   [2., 1., 0., 0.],
                   [1., 0., 2., 3.]])             # same 4x4 image
print("image shape:", t3_img.shape)              # -> image shape: (4, 4)
t3_kernel = np.array([[1., 0.], [0., -1.]])      # same 2x2 kernel
t3_cols = []                                     # rows will be flattened windows
for t3_i in range(3):
    for t3_j in range(3):
        t3_cols.append(t3_img[t3_i:t3_i+2, t3_j:t3_j+2].reshape(-1))
t3_cols = np.array(t3_cols)                      # -> shape (9, 4)
print("im2col shape:", t3_cols.shape)            # -> im2col shape: (9, 4)
print("first flattened window:", t3_cols[0].tolist()) # -> [1.0, 2.0, 0.0, 3.0]
t3_flat_kernel = t3_kernel.reshape(-1)           # -> [1.0, 0.0, 0.0, -1.0]
print("flat kernel:", t3_flat_kernel.tolist())   # -> [1.0, 0.0, 0.0, -1.0]
t3_mm = t3_cols @ t3_flat_kernel                 # -> 9 responses
t3_map = t3_mm.reshape(3, 3)                     # -> [[-2,1,-2],[-1,3,1],[2,-1,-3]]
print("matrix-multiply map:", t3_map.tolist())   # -> [[-2.0, 1.0, -2.0], [-1.0, 3.0, 1.0], [2.0, -1.0, -3.0]]
assert t3_cols.shape == (9, 4) and np.array_equal(t3_map, np.array([[-2., 1., -2.], [-1., 3., 1.], [2., -1., -3.]]))

plt.figure(figsize=(4.4, 2.8))
plt.imshow(t3_cols, cmap="viridis", aspect="auto")
plt.colorbar(label="window value")
plt.xlabel("flattened kernel coordinate")
plt.ylabel("window row")
plt.title("Toy 3 · im2col window matrix")
plt.show()

▶ What you'll see: nine flattened windows multiplied by one flattened kernel recreate the feature map.

### ✍️ Toy 4 · Padding and stride choose legal landing positions

The output size formula counts where the kernel can start on the padded canvas, stepping by the stride.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)                 # seeded generator for this toy
t4_n = 5                                         # -> input length
t4_k = 3                                         # -> kernel length
t4_p = 1                                         # one zero on each side
t4_s = 2                                         # -> stride two
t4_padded = t4_n + 2 * t4_p                      # -> 7
print("padded length:", t4_padded)               # -> 7
t4_last = t4_padded - t4_k                       # -> 4
print("last legal start:", t4_last)              # -> 4
t4_starts = np.arange(0, t4_last + 1, t4_s)      # -> [0, 2, 4]
print("stride starts:", t4_starts.tolist())      # -> [0, 2, 4]
t4_out = (t4_n + 2 * t4_p - t4_k) // t4_s + 1    # -> 3
print("output length:", t4_out)                  # -> 3
assert t4_out == len(t4_starts) == 3

plt.figure(figsize=(4.8, 1.9))
plt.scatter(np.arange(t4_last + 1), np.zeros(t4_last + 1), color="lightgray", s=120, label="legal starts")
plt.scatter(t4_starts, np.zeros_like(t4_starts), color="seagreen", s=170, label="stride starts")
plt.yticks([])
plt.xticks(np.arange(t4_padded))
plt.legend(loc="upper right")
plt.title("Toy 4 · stride samples legal starts")
plt.show()

▶ What you'll see: starts `0`, `2`, and `4` become the three output cells.

### ✍️ Toy 5 · Weight sharing gives translation equivariance

When the input pattern shifts right, the strongest response shifts right because the same kernel is
used everywhere.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)                 # seeded generator for this toy
t5_base = np.zeros((5, 5))                        # -> blank image
t5_base[2, 1] = 1.0                              # -> left shoulder
t5_base[2, 2] = 2.0                              # center peak
t5_base[2, 3] = 1.0                              # -> right shoulder
print("base image:", t5_base.tolist())           # -> [[0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 2.0, 1.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0]]
t5_kernel = np.array([[0., 0., 0.],
                      [1., 1., 1.],
                      [0., 0., 0.]])              # -> horizontal three-pixel detector
print("kernel:", t5_kernel.tolist())             # -> [[0.0, 0.0, 0.0], [1.0, 1.0, 1.0], [0.0, 0.0, 0.0]]
def t5_conv2d(t5_x, t5_k):
    t5_out = np.zeros((t5_x.shape[0] - t5_k.shape[0] + 1, t5_x.shape[1] - t5_k.shape[1] + 1))
    for t5_i in range(t5_out.shape[0]):
        for t5_j in range(t5_out.shape[1]):
            t5_out[t5_i, t5_j] = np.sum(t5_x[t5_i:t5_i+3, t5_j:t5_j+3] * t5_k)
    return t5_out
t5_feature = t5_conv2d(t5_base, t5_kernel)       # -> strongest at (1,1)
print("original feature:", t5_feature.tolist())  # -> [[0.0, 0.0, 0.0], [3.0, 4.0, 3.0], [0.0, 0.0, 0.0]]
t5_shifted = np.zeros_like(t5_base)              # -> blank shifted image
t5_shifted[:, 1:] = t5_base[:, :-1]              # -> shift input one column right
t5_feature_shifted = t5_conv2d(t5_shifted, t5_kernel) # -> strongest at (1,2)
print("shifted feature:", t5_feature_shifted.tolist()) # -> [[0.0, 0.0, 0.0], [1.0, 3.0, 4.0], [0.0, 0.0, 0.0]]
t5_peak = np.unravel_index(np.argmax(t5_feature), t5_feature.shape) # -> (1, 1)
t5_shift_peak = np.unravel_index(np.argmax(t5_feature_shifted), t5_feature_shifted.shape) # -> (1, 2)
print("peak before/after:", t5_peak, t5_shift_peak) # -> (1, 1) (1, 2)
assert t5_shift_peak == (t5_peak[0], t5_peak[1] + 1)

fig, t5_ax = plt.subplots(1, 2, figsize=(5.6, 2.5))
t5_ax[0].imshow(t5_feature, cmap="magma")
t5_ax[0].set_title("original")
t5_ax[1].imshow(t5_feature_shifted, cmap="magma")
t5_ax[1].set_title("shifted")
plt.suptitle("Toy 5 · feature moves with input")
plt.show()

▶ What you'll see: the maximum response moves one output column to the right.

### ✍️ Toy 6 · Correlation and true convolution differ by kernel flipping

Deep-learning “convolution” usually means cross-correlation. True mathematical convolution flips the
kernel first, which changes the answer for asymmetric kernels.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)                 # seeded generator for this toy
t6_patch = np.array([[1., 2.], [3., 4.]])        # one 2x2 patch
print("patch:", t6_patch.tolist())               # -> [[1.0, 2.0], [3.0, 4.0]]
t6_kernel = np.array([[1., 2.], [0., -1.]])      # -> asymmetric kernel
print("kernel:", t6_kernel.tolist())             # -> [[1.0, 2.0], [0.0, -1.0]]
t6_corr_products = t6_patch * t6_kernel          # -> [[1,4],[0,-4]]
print("correlation products:", t6_corr_products.tolist()) # -> [[1.0, 4.0], [0.0, -4.0]]
t6_corr = float(t6_corr_products.sum())          # -> 1.0
print("correlation response:", t6_corr)          # -> 1.0
t6_flipped = np.flip(t6_kernel)                  # -> [[-1,0],[2,1]]
print("flipped kernel:", t6_flipped.tolist())    # -> [[-1.0, 0.0], [2.0, 1.0]]
t6_conv = float(np.sum(t6_patch * t6_flipped))   # -> 9.0
print("true convolution response:", t6_conv)     # -> 9.0
assert t6_corr == 1.0 and t6_conv == 9.0

plt.figure(figsize=(4.4, 2.7))
plt.bar(["correlation", "true conv"], [t6_corr, t6_conv], color=["steelblue", "darkorange"])
plt.ylabel("response")
plt.title("Toy 6 · flipping changes asymmetric kernels")
plt.show()

▶ What you'll see: the unflipped response is `1`, while the flipped-kernel response is `9`.

### ✍️ Toy 7 · Channels and multiple kernels create a feature volume

One filter spans all input channels to make one output map. Multiple filters stack multiple maps along
a new output-channel axis.

In [ ]:
import numpy as np

t7_rng = np.random.default_rng(0)                 # seeded generator for this toy
t7_x = np.zeros((3, 3, 2))                        # -> 3x3 image with 2 channels
t7_x[:, :, 0] = np.array([[1, 0, 1], [0, 1, 0], [1, 0, 1]], dtype=float) # -> checker channel
t7_x[:, :, 1] = np.array([[0, 2, 0], [2, 0, 2], [0, 2, 0]], dtype=float) # -> cross channel
print("input shape:", t7_x.shape)                # -> input shape: (3, 3, 2)
t7_f1 = np.zeros((2, 2, 2))                      # -> first 2x2x2 filter
t7_f1[:, :, 0] = np.array([[1., 0.], [0., 1.]])  # -> diagonal on channel 0
t7_f1[:, :, 1] = np.array([[0., 1.], [1., 0.]])  # -> anti-diagonal on channel 1
print("filter shape:", t7_f1.shape)              # -> filter shape: (2, 2, 2)
t7_f2 = -t7_f1                                   # -> second filter is the negative
t7_maps = []                                     # one map per filter
for t7_filter in [t7_f1, t7_f2]:
    t7_out = np.zeros((2, 2))
    for t7_i in range(2):
        for t7_j in range(2):
            t7_out[t7_i, t7_j] = np.sum(t7_x[t7_i:t7_i+2, t7_j:t7_j+2, :] * t7_filter)
    t7_maps.append(t7_out)
t7_volume = np.stack(t7_maps, axis=-1)           # -> shape (2,2,2)
print("first output map:", t7_volume[:, :, 0].tolist()) # -> [[6.0, 0.0], [0.0, 6.0]]
print("second output map:", t7_volume[:, :, 1].tolist()) # -> [[-6.0, 0.0], [0.0, -6.0]]
print("feature volume shape:", t7_volume.shape)  # -> feature volume shape: (2, 2, 2)
assert t7_volume.shape == (2, 2, 2) and t7_volume[0, 0, 0] == 6.0 and t7_volume[0, 0, 1] == -6.0

fig, t7_ax = plt.subplots(1, 2, figsize=(5.2, 2.4))
t7_ax[0].imshow(t7_volume[:, :, 0], cmap="coolwarm", vmin=-6, vmax=6)
t7_ax[0].set_title("filter 1")
t7_ax[1].imshow(t7_volume[:, :, 1], cmap="coolwarm", vmin=-6, vmax=6)
t7_ax[1].set_title("filter 2")
plt.suptitle("Toy 7 · maps stack into volume")
plt.show()

▶ What you'll see: two filters over the same input produce a `2×2×2` feature volume.

### ✍️ Toy 8 · Parameter sharing replaces many dense weights

A convolution learns one small stencil and reuses it; a dense image-to-map layer would need a separate
weight for every input-output pair.

In [ ]:
import numpy as np

t8_rng = np.random.default_rng(0)                 # seeded generator for this toy
t8_h = 5                                         # -> input height
t8_w = 5                                         # -> input width
t8_k = 3                                         # -> 3x3 kernel
t8_out_h = t8_h - t8_k + 1                       # -> 3
t8_out_w = t8_w - t8_k + 1                       # -> 3
print("valid output shape:", (t8_out_h, t8_out_w)) # -> (3, 3)
t8_conv_params = t8_k * t8_k                     # -> 9
print("shared conv weights:", t8_conv_params)    # -> 9
t8_dense_params = (t8_h * t8_w) * (t8_out_h * t8_out_w) # -> 225
print("dense weights 25→9:", t8_dense_params)    # -> 225
t8_factor = float(t8_dense_params / t8_conv_params) # -> 25.0
print("dense/conv factor:", round(t8_factor, 1)) # -> 25.0
assert t8_conv_params == 9 and t8_dense_params == 225 and t8_factor == 25.0

plt.figure(figsize=(4.6, 2.7))
plt.bar(["3×3 conv", "dense 25→9"], [t8_conv_params, t8_dense_params], color=["teal", "crimson"])
plt.yscale("log")
plt.ylabel("parameters")
plt.title("Toy 8 · sharing saves weights")
plt.show()

▶ What you'll see: the dense layer uses 25 times as many weights in this tiny setup.

### ✍️ Toy 9 · Stacking small kernels grows the receptive field

Each stride-1 `3×3` layer adds one pixel of context on every side, so receptive-field width grows by 2
per layer.

In [ ]:
import numpy as np

t9_rng = np.random.default_rng(0)                 # seeded generator for this toy
t9_layers = np.arange(1, 5)                      # -> [1, 2, 3, 4]
print("layers:", t9_layers.tolist())             # -> [1, 2, 3, 4]
t9_receptive = 1 + 2 * t9_layers                 # -> [3, 5, 7, 9]
print("receptive-field widths:", t9_receptive.tolist()) # -> [3, 5, 7, 9]
t9_added_context = t9_receptive - 1              # -> [2, 4, 6, 8]
print("extra context over one pixel:", t9_added_context.tolist()) # -> [2, 4, 6, 8]
assert np.array_equal(t9_receptive, np.array([3, 5, 7, 9]))

plt.figure(figsize=(4.4, 2.7))
plt.plot(t9_layers, t9_receptive, marker="o", color="purple")
plt.xlabel("number of 3×3 layers")
plt.ylabel("receptive-field width")
plt.title("Toy 9 · depth expands context")
plt.show()

▶ What you'll see: four small-kernel layers see a `9×9` input region.


## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for arrays, padding, dot products, and numerical checks.
import matplotlib.pyplot as plt  # load Matplotlib for heatmaps, bars, and line plots.
np.random.seed(0)  # make examples reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Multiply one patch by one kernel

**Goal.** Compute the smallest convolution response by hand, because every larger feature map is just this dot product repeated. We build it in 2 steps.

In [ ]:
patch_b1 = np.array([[1., 2.], [3., 4.]])  # define one local image window.
kernel_b1 = np.array([[1., 0.], [0., -1.]])  # define a diagonal-difference detector.
products_b1 = patch_b1 * kernel_b1  # multiply matching entries before summing.
print("products:\n", products_b1)

▶ What you'll see: only the top-left and bottom-right entries contribute because the other kernel weights are 0.

In [ ]:
response_b1 = float(products_b1.sum())  # sum elementwise products into one convolution response.
print("response:", response_b1)
assert response_b1 == -3.0
plt.figure(figsize=(3.5, 3))
plt.imshow(products_b1, cmap="coolwarm", vmin=-4, vmax=4)
plt.colorbar(label="product")
plt.title("Basic 1: terms in one dot product")
plt.show()

▶ What you'll see: the −4 contribution dominates, so the response is −3.

👀 Takeaway: one convolution output is one kernel-window dot product.

### Basic 2 — Slide over a 3×3 ramp

**Goal.** Slide a 2×2 kernel over all valid positions, because a feature map is a grid of repeated local dot products. We build it in 2 steps.

In [ ]:
x_b2 = np.array([[1., 2., 3.], [4., 5., 6.], [7., 8., 9.]])  # a simple ramp image.
k_b2 = np.array([[1., 0.], [0., -1.]])  # compare top-left against bottom-right.
out_b2 = np.zeros((2, 2))  # valid 3×3 with 2×2 kernel gives a 2×2 map.
for i_b2 in range(2):
    for j_b2 in range(2):
        out_b2[i_b2, j_b2] = np.sum(x_b2[i_b2:i_b2 + 2, j_b2:j_b2 + 2] * k_b2)
print(out_b2)
assert np.all(out_b2 == -4.0)

▶ What you'll see: every valid position gives −4 because the ramp rises by the same diagonal amount everywhere.

In [ ]:
plt.figure(figsize=(3.6, 3))
plt.imshow(out_b2, cmap="coolwarm", vmin=-5, vmax=0)
plt.colorbar(label="response")
plt.title("Basic 2: constant diagonal response")
plt.show()

▶ What you'll see: the feature map is uniform, which is correct for a linear ramp.

👀 Takeaway: constant feature maps can be a meaningful signal, not a bug.

### Basic 3 — Confirm valid output shape

**Goal.** Count landing positions for a valid convolution, because output shape follows from where the kernel fits completely inside the input. We build it in 2 steps.

In [ ]:
H_b3, W_b3 = 5, 6  # input height and width.
kh_b3, kw_b3 = 3, 2  # kernel height and width.
out_shape_b3 = (H_b3 - kh_b3 + 1, W_b3 - kw_b3 + 1)  # valid convolution shape.
print("valid output shape:", out_shape_b3)
assert out_shape_b3 == (3, 5)

▶ What you'll see: a 3×2 kernel can land in 3 vertical and 5 horizontal valid positions.

In [ ]:
canvas_b3 = np.zeros((H_b3, W_b3))  # create a blank input canvas.
canvas_b3[:kh_b3, :kw_b3] = 1  # mark the first kernel footprint.
canvas_b3[H_b3-kh_b3:, W_b3-kw_b3:] = 2  # mark the last valid footprint.
plt.figure(figsize=(4, 3))
plt.imshow(canvas_b3, cmap="viridis")
plt.title("Basic 3: first and last valid footprints")
plt.show()

▶ What you'll see: the first and last footprints show why only 3×5 starting positions are legal.

👀 Takeaway: valid convolution shrinks each dimension by kernel_size − 1.

### Basic 4 — Add zero padding

**Goal.** Pad a small image by one pixel, because padding lets edge pixels participate in centered local windows. We build it in 2 steps.

In [ ]:
x_b4 = np.arange(1, 10, dtype=float).reshape(3, 3)  # a 3×3 image with visible values.
pad_b4 = 1  # add one zero border on every side.
xp_b4 = np.pad(x_b4, ((pad_b4, pad_b4), (pad_b4, pad_b4)), mode="constant")
print("padded shape:", xp_b4.shape)
print(xp_b4)
assert xp_b4.shape == (5, 5)

▶ What you'll see: the original 3×3 grid is surrounded by zeros.

In [ ]:
plt.figure(figsize=(3.8, 3))
plt.imshow(xp_b4, cmap="gray")
plt.title("Basic 4: zero-padded image")
plt.colorbar(label="value")
plt.show()

▶ What you'll see: the border is dark because padding inserted zeros.

👀 Takeaway: padding changes the working canvas before the kernel slides.

### Basic 5 — Use stride two

**Goal.** Skip landing positions with stride 2, because strided convolution deliberately lowers spatial resolution. We build it in 2 steps.

In [ ]:
x_b5 = np.arange(1, 26, dtype=float).reshape(5, 5)  # a 5×5 input.
k_b5 = np.ones((2, 2))  # sum each 2×2 window.
starts_b5 = [(i_b5, j_b5) for i_b5 in range(0, 5 - 2 + 1, 2) for j_b5 in range(0, 5 - 2 + 1, 2)]
print("landing starts:", starts_b5)
assert starts_b5 == [(0, 0), (0, 2), (2, 0), (2, 2)]

▶ What you'll see: stride 2 lands at rows and columns 0 and 2, skipping the in-between starts.

In [ ]:
out_b5 = np.array([[np.sum(x_b5[i_b5:i_b5 + 2, j_b5:j_b5 + 2] * k_b5) for j_b5 in range(0, 4, 2)] for i_b5 in range(0, 4, 2)])
print(out_b5)
assert out_b5.shape == (2, 2)
plt.figure(figsize=(3.6, 3))
plt.imshow(out_b5, cmap="magma")
plt.colorbar(label="2×2 sum")
plt.title("Basic 5: stride-2 outputs")
plt.show()

▶ What you'll see: only four windows are evaluated, so the output is 2×2.

👀 Takeaway: stride reduces resolution by evaluating fewer kernel positions.

### Basic 6 — Detect a vertical edge

**Goal.** Apply a simple left-minus-right kernel, because many classic image filters are hand-coded convolutions. We build it in 2 steps.

In [ ]:
img_b6 = np.zeros((5, 5))
img_b6[:, 3:] = 1.0  # create a sharp vertical step from dark to bright.
k_b6 = np.array([[-1., 1.]])  # compare right pixel against left pixel.
out_b6 = np.zeros((5, 4))
for i_b6 in range(5):
    for j_b6 in range(4):
        out_b6[i_b6, j_b6] = np.sum(img_b6[i_b6:i_b6 + 1, j_b6:j_b6 + 2] * k_b6)
print(out_b6[0])
assert out_b6[0, 2] == 1.0

▶ What you'll see: the response is nonzero where the window crosses the dark-to-bright boundary.

In [ ]:
fig_b6, ax_b6 = plt.subplots(1, 2, figsize=(6, 2.6))
ax_b6[0].imshow(img_b6, cmap="gray", vmin=0, vmax=1); ax_b6[0].set_title("input step")
ax_b6[1].imshow(out_b6, cmap="coolwarm", vmin=-1, vmax=1); ax_b6[1].set_title("edge response")
plt.suptitle("Basic 6: vertical edge detector"); plt.show()

▶ What you'll see: a bright response line appears exactly at the vertical edge.

👀 Takeaway: convolution kernels can encode local contrasts such as edges.

### Basic 7 — Separate positive and negative responses

**Goal.** Inspect signed responses, because kernels can reward one orientation and penalize the opposite orientation. We build it in 2 steps.

In [ ]:
row_b7 = np.array([[0., 0., 1., 1., 0., 0.]])  # bright block with rising and falling edges.
k_b7 = np.array([[-1., 1.]])  # positive for rising, negative for falling.
resp_b7 = np.array([np.sum(row_b7[:, j_b7:j_b7 + 2] * k_b7) for j_b7 in range(row_b7.shape[1] - 1)])
print("responses:", resp_b7)
assert np.array_equal(resp_b7, np.array([0., 1., 0., -1., 0.]))

▶ What you'll see: the left edge is +1 and the right edge is −1.

In [ ]:
plt.figure(figsize=(5, 2.8))
plt.bar(range(len(resp_b7)), resp_b7, color=["gray" if v_b7 == 0 else "seagreen" if v_b7 > 0 else "crimson" for v_b7 in resp_b7])
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Basic 7: signed edge responses")
plt.xlabel("window start")
plt.ylabel("response")
plt.show()

▶ What you'll see: positive and negative bars mark opposite transitions.

👀 Takeaway: response sign matters when the kernel has positive and negative weights.

### Basic 8 — Compare correlation and true convolution

**Goal.** Flip a kernel before applying it, because deep-learning convolution usually uses cross-correlation rather than the flipped mathematical convolution. We build it in 2 steps.

In [ ]:
x_b8 = np.array([[1., 2., 3.]])  # one-dimensional signal stored as one row.
k_b8 = np.array([[1., 2.]])  # asymmetric kernel so flipping changes the result.
corr_b8 = np.array([np.sum(x_b8[:, j_b8:j_b8 + 2] * k_b8) for j_b8 in range(2)])
true_conv_b8 = np.array([np.sum(x_b8[:, j_b8:j_b8 + 2] * np.flip(k_b8, axis=1)) for j_b8 in range(2)])
print("cross-correlation:", corr_b8)
print("true convolution:", true_conv_b8)
assert np.array_equal(corr_b8, np.array([5., 8.])) and np.array_equal(true_conv_b8, np.array([4., 7.]))

▶ What you'll see: the two outputs differ because the asymmetric kernel was flipped only for true convolution.

In [ ]:
plt.figure(figsize=(4.2, 3))
plt.plot(corr_b8, marker="o", label="DL correlation")
plt.plot(true_conv_b8, marker="s", label="flipped convolution")
plt.title("Basic 8: flip changes asymmetric kernels")
plt.legend()
plt.show()

▶ What you'll see: the lines are offset, proving the convention matters for fixed hand-designed kernels.

👀 Takeaway: CNN libraries usually learn unflipped kernels, so they implement cross-correlation by signal-processing terminology.

### Basic 9 — Sum across channels

**Goal.** Combine channel-specific products into one response, because a color-image filter has one slice per input channel. We build it in 2 steps.

In [ ]:
patch_b9 = np.zeros((2, 2, 3))
patch_b9[:, :, 0] = [[1., 0.], [0., 1.]]
patch_b9[:, :, 1] = [[0., 1.], [1., 0.]]
patch_b9[:, :, 2] = 0.5
filter_b9 = np.ones((2, 2, 3))
filter_b9[:, :, 1] *= -1  # oppose the green channel.
channel_sums_b9 = np.array([np.sum(patch_b9[:, :, c_b9] * filter_b9[:, :, c_b9]) for c_b9 in range(3)])
print("channel contributions:", channel_sums_b9)
assert np.array_equal(channel_sums_b9, np.array([2., -2., 2.]))

▶ What you'll see: red contributes +2, green contributes −2, and blue contributes +2.

In [ ]:
response_b9 = float(channel_sums_b9.sum())
print("multi-channel response:", response_b9)
assert response_b9 == 2.0
plt.figure(figsize=(4, 3))
plt.bar(["red", "green", "blue"], channel_sums_b9, color=["red", "green", "blue"])
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Basic 9: channel contributions")
plt.show()

▶ What you'll see: the final response is the sum of all channel contributions.

👀 Takeaway: multi-channel convolution sums over height, width, and input-channel axes.

### Basic 10 — Count shared weights

**Goal.** Compare convolution and dense parameter counts, because weight sharing is the main efficiency bargain. We build it in 2 steps.

In [ ]:
input_pixels_b10 = 32 * 32  # a small grayscale image.
output_pixels_b10 = 30 * 30  # valid 3×3 output positions.
conv_weights_b10 = 3 * 3  # one shared 3×3 kernel.
dense_weights_b10 = input_pixels_b10 * output_pixels_b10  # one independent weight per input-output pair.
print("conv weights:", conv_weights_b10, "dense weights:", dense_weights_b10)
assert conv_weights_b10 == 9 and dense_weights_b10 == 921600

▶ What you'll see: the dense layer would spend 921,600 weights where one convolution kernel spends 9.

In [ ]:
plt.figure(figsize=(4.2, 3))
plt.bar(["shared conv", "dense"], [conv_weights_b10, dense_weights_b10], color=["teal", "crimson"])
plt.yscale("log")
plt.ylabel("number of weights")
plt.title("Basic 10: parameter sharing")
plt.show()

▶ What you'll see: the dense count is orders of magnitude larger even for a tiny image.

👀 Takeaway: convolution is cheap because the same small kernel is reused at every output location.

## 🟡 Easy

### Easy 1 — Implement a reusable 2D convolution

**Goal.** Write a small valid 2D convolution function, because the whole layer is nested loops plus one local sum. We build it in 3 steps.

In [ ]:
x_e1 = np.arange(1, 17, dtype=float).reshape(4, 4)  # a 4×4 test image.
k_e1 = np.array([[1., 0.], [0., -1.]])  # diagonal-difference kernel.
print("input:\n", x_e1)
print("kernel:\n", k_e1)

▶ What you'll see: the input rises steadily, so this diagonal difference should be constant.

In [ ]:
def conv_valid_e1(x_e1, k_e1):
    out_e1 = np.zeros((x_e1.shape[0] - k_e1.shape[0] + 1, x_e1.shape[1] - k_e1.shape[1] + 1))
    for i_e1 in range(out_e1.shape[0]):
        for j_e1 in range(out_e1.shape[1]):
            out_e1[i_e1, j_e1] = np.sum(x_e1[i_e1:i_e1 + k_e1.shape[0], j_e1:j_e1 + k_e1.shape[1]] * k_e1)
    return out_e1

out_e1 = conv_valid_e1(x_e1, k_e1)
print(out_e1)
assert out_e1.shape == (3, 3)
assert np.all(out_e1 == -5.0)

In [ ]:
plt.figure(figsize=(4, 3))
plt.imshow(out_e1, cmap="coolwarm", vmin=-6, vmax=0)
plt.colorbar(label="response")
plt.title("Easy 1: valid convolution output")
plt.show()

▶ What you'll see: every output is −5 because each 2×2 diagonal increases by five.

👀 Takeaway: a convolution function is just systematic extraction, multiplication, and summation.

### Easy 2 — Verify the stride/padding formula

**Goal.** Compute outputs for several stride and padding choices, because shape mistakes are common CNN bugs. We build it in 3 steps.

In [ ]:
n_e2 = 7  # one spatial dimension of the input.
k_e2 = 3  # kernel width.
configs_e2 = [(0, 1), (1, 1), (0, 2), (2, 2)]  # (padding, stride) cases.
sizes_e2 = []
for p_e2, s_e2 in configs_e2:
    sizes_e2.append((n_e2 + 2 * p_e2 - k_e2) // s_e2 + 1)
print("sizes:", sizes_e2)
assert sizes_e2 == [5, 7, 3, 5]

▶ What you'll see: padding can preserve or enlarge size, while stride reduces the number of landings.

In [ ]:
labels_e2 = [f"p={p_e2},s={s_e2}" for p_e2, s_e2 in configs_e2]
plt.figure(figsize=(5, 3))
plt.bar(labels_e2, sizes_e2, color="slateblue")
plt.ylabel("1D output size")
plt.title("Easy 2: output-size formula")
plt.show()

In [ ]:
manual_starts_e2 = list(range(0, n_e2 + 2 * 0 - k_e2 + 1, 2))
print("stride-2 starts without padding:", manual_starts_e2)
assert manual_starts_e2 == [0, 2, 4]

▶ What you'll see: the floor in the formula corresponds to the finite list of legal starting positions.

👀 Takeaway: output shape is bookkeeping over legal kernel starts, not a separate learned behavior.

### Easy 3 — Apply two different kernels to one image

**Goal.** Create a two-channel feature volume, because CNN layers learn many detectors at once. We build it in 3 steps.

In [ ]:
img_e3 = np.zeros((6, 6))
img_e3[2:4, :] = 1.0  # horizontal stripe.
img_e3[:, 4:] += 1.0  # vertical bright region.
kh_e3 = np.array([[-1., -1.], [1., 1.]])  # horizontal-change detector.
kv_e3 = np.array([[-1., 1.], [-1., 1.]])  # vertical-change detector.
print("image max:", img_e3.max())
assert img_e3.shape == (6, 6)

▶ What you'll see: the image contains both horizontal and vertical structure.

In [ ]:
def conv_valid_e3(x_e3, k_e3):
    out_e3 = np.zeros((x_e3.shape[0] - k_e3.shape[0] + 1, x_e3.shape[1] - k_e3.shape[1] + 1))
    for i_e3 in range(out_e3.shape[0]):
        for j_e3 in range(out_e3.shape[1]):
            out_e3[i_e3, j_e3] = np.sum(x_e3[i_e3:i_e3 + k_e3.shape[0], j_e3:j_e3 + k_e3.shape[1]] * k_e3)
    return out_e3

feat_h_e3 = conv_valid_e3(img_e3, kh_e3)
feat_v_e3 = conv_valid_e3(img_e3, kv_e3)
volume_e3 = np.stack([feat_h_e3, feat_v_e3], axis=-1)
print("feature volume shape:", volume_e3.shape)
assert volume_e3.shape == (5, 5, 2)

In [ ]:
fig_e3, ax_e3 = plt.subplots(1, 2, figsize=(6, 2.8))
ax_e3[0].imshow(feat_h_e3, cmap="coolwarm"); ax_e3[0].set_title("horizontal change")
ax_e3[1].imshow(feat_v_e3, cmap="coolwarm"); ax_e3[1].set_title("vertical change")
plt.suptitle("Easy 3: two kernels, two maps")
plt.show()

▶ What you'll see: the two maps light up on different structures in the same input.

👀 Takeaway: output channels correspond to different learned kernels applied to the same spatial grid.

### Easy 4 — Show translation equivariance numerically

**Goal.** Shift an input pattern and confirm the response shifts too, because shared weights make location behavior consistent. We build it in 3 steps.

In [ ]:
img_e4 = np.zeros((6, 6))
img_e4[1:3, 1:3] = 1.0  # small square near the upper-left.
k_e4 = np.ones((2, 2))  # detector for a filled 2×2 square.
img_shift_e4 = np.zeros_like(img_e4)
img_shift_e4[2:4, 2:4] = 1.0  # same square shifted down-right.
print("sum original:", img_e4.sum(), "sum shifted:", img_shift_e4.sum())
assert img_e4.sum() == img_shift_e4.sum() == 4.0

▶ What you'll see: both images contain the same pattern with a different position.

In [ ]:
def conv_valid_e4(x_e4, k_e4):
    out_e4 = np.zeros((x_e4.shape[0] - k_e4.shape[0] + 1, x_e4.shape[1] - k_e4.shape[1] + 1))
    for i_e4 in range(out_e4.shape[0]):
        for j_e4 in range(out_e4.shape[1]):
            out_e4[i_e4, j_e4] = np.sum(x_e4[i_e4:i_e4 + k_e4.shape[0], j_e4:j_e4 + k_e4.shape[1]] * k_e4)
    return out_e4

resp_e4 = conv_valid_e4(img_e4, k_e4)
resp_shift_e4 = conv_valid_e4(img_shift_e4, k_e4)
print("original max location:", np.unravel_index(np.argmax(resp_e4), resp_e4.shape))
print("shifted max location:", np.unravel_index(np.argmax(resp_shift_e4), resp_shift_e4.shape))
assert np.unravel_index(np.argmax(resp_shift_e4), resp_shift_e4.shape) == (2, 2)

In [ ]:
fig_e4, ax_e4 = plt.subplots(1, 2, figsize=(6, 2.8))
ax_e4[0].imshow(resp_e4, cmap="viridis", vmin=0, vmax=4); ax_e4[0].set_title("original response")
ax_e4[1].imshow(resp_shift_e4, cmap="viridis", vmin=0, vmax=4); ax_e4[1].set_title("shifted response")
plt.suptitle("Easy 4: response moves with pattern")
plt.show()

▶ What you'll see: the maximum response moves down and right with the square.

👀 Takeaway: convolution is translation-equivariant because each location uses the same kernel weights.

### Easy 5 — Stack two 3×3 layers and trace the receptive field

**Goal.** Mark which input pixels influence one final output, because stacking local operations grows context. We build it in 3 steps.

In [ ]:
input_mask_e5 = np.zeros((5, 5))
input_mask_e5[2, 2] = 1.0  # one input impulse in the center.
k_e5 = np.ones((3, 3))  # any nonzero 3×3 kernel spreads influence locally.
print("input impulse location:", np.argwhere(input_mask_e5 == 1)[0])
assert input_mask_e5.sum() == 1.0

▶ What you'll see: one central input pixel is the only initial nonzero evidence.

In [ ]:
def conv_full_e5(x_e5, k_e5):
    xp_e5 = np.pad(x_e5, ((1, 1), (1, 1)), mode="constant")
    out_e5 = np.zeros_like(x_e5)
    for i_e5 in range(x_e5.shape[0]):
        for j_e5 in range(x_e5.shape[1]):
            out_e5[i_e5, j_e5] = np.sum(xp_e5[i_e5:i_e5 + 3, j_e5:j_e5 + 3] * k_e5)
    return out_e5

layer1_e5 = conv_full_e5(input_mask_e5, k_e5)
layer2_e5 = conv_full_e5(layer1_e5, k_e5)
print("nonzero after one layer:", int(np.sum(layer1_e5 > 0)))
print("nonzero after two layers:", int(np.sum(layer2_e5 > 0)))
assert int(np.sum(layer1_e5 > 0)) == 9 and int(np.sum(layer2_e5 > 0)) == 25

In [ ]:
fig_e5, ax_e5 = plt.subplots(1, 2, figsize=(6, 2.8))
ax_e5[0].imshow(layer1_e5 > 0, cmap="gray"); ax_e5[0].set_title("after 1 layer")
ax_e5[1].imshow(layer2_e5 > 0, cmap="gray"); ax_e5[1].set_title("after 2 layers")
plt.suptitle("Easy 5: receptive field expansion")
plt.show()

▶ What you'll see: the affected region grows from 3×3 to 5×5.

👀 Takeaway: depth expands receptive field while each layer still uses small local kernels.

## 🔴 Advanced

### Advanced 1 — Build im2col and matrix multiplication

**Goal.** Rewrite convolution as a matrix multiplication, because efficient implementations batch all local dot products. We build it in 4 steps.

In [ ]:
x_a1 = np.arange(1, 17, dtype=float).reshape(4, 4)  # input image.
k_a1 = np.array([[1., 0.], [0., -1.]])  # 2×2 kernel.
patches_a1 = []
for i_a1 in range(3):
    for j_a1 in range(3):
        patches_a1.append(x_a1[i_a1:i_a1 + 2, j_a1:j_a1 + 2].ravel())
cols_a1 = np.vstack(patches_a1)
print("im2col shape:", cols_a1.shape)
assert cols_a1.shape == (9, 4)

▶ What you'll see: nine flattened patches become nine rows of a design matrix.

In [ ]:
kernel_vec_a1 = k_a1.ravel()
flat_out_a1 = cols_a1 @ kernel_vec_a1
out_a1 = flat_out_a1.reshape(3, 3)
print(out_a1)
assert np.all(out_a1 == -5.0)

In [ ]:
loop_out_a1 = np.zeros((3, 3))
for i_a1 in range(3):
    for j_a1 in range(3):
        loop_out_a1[i_a1, j_a1] = np.sum(x_a1[i_a1:i_a1 + 2, j_a1:j_a1 + 2] * k_a1)
print("matches loop:", np.allclose(out_a1, loop_out_a1))
assert np.allclose(out_a1, loop_out_a1)

In [ ]:
plt.figure(figsize=(4.2, 3))
plt.imshow(cols_a1, cmap="viridis", aspect="auto")
plt.colorbar(label="patch entry")
plt.title("Advanced 1: im2col patch matrix")
plt.xlabel("flattened kernel coordinate")
plt.ylabel("output location")
plt.show()

▶ What you'll see: each row is one local window; multiplying by the flattened kernel computes all responses at once.

👀 Takeaway: convolution can be vectorized as many local dot products in one matrix multiply.

### Advanced 2 — Compute a multi-channel, multi-kernel layer

**Goal.** Implement a tiny convolution layer with input channels and output channels, because CNN tensors are feature volumes, not just matrices. We build it in 4 steps.

In [ ]:
x_a2 = np.zeros((4, 4, 2))
x_a2[:, :, 0] = np.arange(16).reshape(4, 4) / 10.0
x_a2[:, :, 1] = np.flipud(x_a2[:, :, 0])
W_a2 = np.zeros((2, 2, 2, 3))  # kh, kw, in_channels, out_channels.
W_a2[:, :, 0, 0] = 1.0
W_a2[:, :, 1, 1] = -1.0
W_a2[:, :, :, 2] = 0.5
print("x shape:", x_a2.shape, "W shape:", W_a2.shape)
assert W_a2.shape == (2, 2, 2, 3)

▶ What you'll see: each of three output kernels has a 2×2 slice for each of two input channels.

In [ ]:
out_a2 = np.zeros((3, 3, 3))
for i_a2 in range(3):
    for j_a2 in range(3):
        patch_a2 = x_a2[i_a2:i_a2 + 2, j_a2:j_a2 + 2, :]
        for c_a2 in range(3):
            out_a2[i_a2, j_a2, c_a2] = np.sum(patch_a2 * W_a2[:, :, :, c_a2])
print("output shape:", out_a2.shape)
assert out_a2.shape == (3, 3, 3)

In [ ]:
print("top-left output vector:", np.round(out_a2[0, 0], 3))
assert np.allclose(np.round(out_a2[0, 0], 3), np.array([1.0, -4.2, 2.6]))

In [ ]:
fig_a2, ax_a2 = plt.subplots(1, 3, figsize=(8, 2.6))
for c_a2 in range(3):
    ax_a2[c_a2].imshow(out_a2[:, :, c_a2], cmap="coolwarm")
    ax_a2[c_a2].set_title(f"out ch {c_a2}")
plt.suptitle("Advanced 2: three output channels")
plt.show()

▶ What you'll see: each output channel has a different response pattern because each kernel mixes channels differently.

👀 Takeaway: output depth equals the number of kernels, while each kernel spans all input channels.

### Advanced 3 — Compare valid, same, and strided outputs

**Goal.** Run one kernel under three shape policies, because architecture choices trade border coverage for resolution. We build it in 4 steps.

In [ ]:
x_a3 = np.zeros((7, 7))
x_a3[2:5, 2:5] = 1.0  # centered square.
k_a3 = np.ones((3, 3)) / 9.0  # local average kernel.
print("input sum:", x_a3.sum(), "kernel sum:", k_a3.sum())
assert x_a3.sum() == 9.0 and round(float(k_a3.sum()), 3) == 1.0

▶ What you'll see: the kernel is an averaging filter, so responses are local fractions of the square.

In [ ]:
def conv_policy_a3(x_a3, k_a3, pad_a3=0, stride_a3=1):
    xp_a3 = np.pad(x_a3, ((pad_a3, pad_a3), (pad_a3, pad_a3)), mode="constant")
    oh_a3 = (xp_a3.shape[0] - k_a3.shape[0]) // stride_a3 + 1
    ow_a3 = (xp_a3.shape[1] - k_a3.shape[1]) // stride_a3 + 1
    out_a3 = np.zeros((oh_a3, ow_a3))
    for i_a3 in range(oh_a3):
        for j_a3 in range(ow_a3):
            r_a3, c_a3 = i_a3 * stride_a3, j_a3 * stride_a3
            out_a3[i_a3, j_a3] = np.sum(xp_a3[r_a3:r_a3 + 3, c_a3:c_a3 + 3] * k_a3)
    return out_a3

valid_a3 = conv_policy_a3(x_a3, k_a3, pad_a3=0, stride_a3=1)
same_a3 = conv_policy_a3(x_a3, k_a3, pad_a3=1, stride_a3=1)
stride_a3 = conv_policy_a3(x_a3, k_a3, pad_a3=0, stride_a3=2)
print("shapes:", valid_a3.shape, same_a3.shape, stride_a3.shape)
assert valid_a3.shape == (5, 5) and same_a3.shape == (7, 7) and stride_a3.shape == (3, 3)

In [ ]:
print("center responses:", round(valid_a3[2, 2], 3), round(same_a3[3, 3], 3), round(stride_a3[1, 1], 3))
assert round(valid_a3[2, 2], 3) == 1.0 and round(stride_a3[1, 1], 3) == 1.0

In [ ]:
fig_a3, ax_a3 = plt.subplots(1, 3, figsize=(8, 2.6))
for data_a3, title_a3, axis_a3 in [(valid_a3, "valid", ax_a3[0]), (same_a3, "same pad", ax_a3[1]), (stride_a3, "stride 2", ax_a3[2])]:
    axis_a3.imshow(data_a3, cmap="viridis", vmin=0, vmax=1)
    axis_a3.set_title(title_a3)
plt.suptitle("Advanced 3: same kernel, different grids")
plt.show()

▶ What you'll see: same padding keeps the largest grid, while stride 2 keeps only a coarse sample of responses.

👀 Takeaway: padding and stride change the output lattice, not the meaning of one local response.

### Advanced 4 — Demonstrate boundary effects from padding

**Goal.** Compare zero padding and reflect padding, because border values can change edge responses even with the same kernel. We build it in 4 steps.

In [ ]:
x_a4 = np.ones((5, 5))  # a constant image should have zero gradient inside.
k_a4 = np.array([[0., 1., 0.], [1., -4., 1.], [0., 1., 0.]])  # Laplacian-like contrast kernel.
print("kernel sum:", k_a4.sum())
assert k_a4.sum() == 0.0

▶ What you'll see: a zero-sum kernel should return zero on a perfectly constant region.

In [ ]:
def conv_pad_mode_a4(x_a4, k_a4, mode_a4):
    xp_a4 = np.pad(x_a4, ((1, 1), (1, 1)), mode=mode_a4)
    out_a4 = np.zeros_like(x_a4)
    for i_a4 in range(x_a4.shape[0]):
        for j_a4 in range(x_a4.shape[1]):
            out_a4[i_a4, j_a4] = np.sum(xp_a4[i_a4:i_a4 + 3, j_a4:j_a4 + 3] * k_a4)
    return out_a4

zero_a4 = conv_pad_mode_a4(x_a4, k_a4, "constant")
reflect_a4 = conv_pad_mode_a4(x_a4, k_a4, "reflect")
print("zero-pad corner:", zero_a4[0, 0], "reflect corner:", reflect_a4[0, 0])
assert zero_a4[0, 0] == -2.0 and reflect_a4[0, 0] == 0.0

In [ ]:
print("zero-pad border sum:", zero_a4.sum(), "reflect sum:", reflect_a4.sum())
assert reflect_a4.sum() == 0.0

In [ ]:
fig_a4, ax_a4 = plt.subplots(1, 2, figsize=(6, 2.8))
ax_a4[0].imshow(zero_a4, cmap="coolwarm", vmin=-2, vmax=0); ax_a4[0].set_title("zero padding")
ax_a4[1].imshow(reflect_a4, cmap="coolwarm", vmin=-2, vmax=0); ax_a4[1].set_title("reflect padding")
plt.suptitle("Advanced 4: padding creates border behavior")
plt.show()

▶ What you'll see: zero padding creates artificial negative border responses; reflect padding keeps a constant image constant.

👀 Takeaway: padding is a modeling choice about what exists beyond the image boundary.

### Advanced 5 — Track receptive field with stride

**Goal.** Compute receptive field, jump, and start across layers, because deeper CNN shape reasoning needs more than output size. We build it in 4 steps.

In [ ]:
layers_a5 = [(3, 1), (3, 2), (3, 1)]  # (kernel size, stride) for three convolution layers.
rf_a5 = 1  # one output initially sees one input position.
jump_a5 = 1  # distance in input pixels between adjacent current-layer positions.
history_a5 = []
for k_a5, s_a5 in layers_a5:
    rf_a5 = rf_a5 + (k_a5 - 1) * jump_a5
    jump_a5 = jump_a5 * s_a5
    history_a5.append((rf_a5, jump_a5))
print("(receptive field, jump):", history_a5)
assert history_a5 == [(3, 1), (5, 2), (9, 2)]

▶ What you'll see: stride increases the jump between neighboring final outputs, while kernels expand the receptive field.

In [ ]:
rf_values_a5 = np.array([h_a5[0] for h_a5 in history_a5])
jump_values_a5 = np.array([h_a5[1] for h_a5 in history_a5])
print("final receptive field:", rf_values_a5[-1], "final jump:", jump_values_a5[-1])
assert rf_values_a5[-1] == 9 and jump_values_a5[-1] == 2

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot([1, 2, 3], rf_values_a5, marker="o", label="receptive field")
plt.plot([1, 2, 3], jump_values_a5, marker="s", label="jump")
plt.xticks([1, 2, 3])
plt.xlabel("layer")
plt.title("Advanced 5: receptive-field bookkeeping")
plt.legend()
plt.show()

In [ ]:
covered_positions_a5 = np.arange(rf_values_a5[-1])
print("one final unit covers input offsets:", covered_positions_a5)
assert len(covered_positions_a5) == 9

▶ What you'll see: the final unit covers a 9-wide input span, and adjacent final units are spaced 2 input pixels apart.

👀 Takeaway: stride changes both resolution and how receptive fields are spaced in the original image.